In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
sys.path.insert(0, os.path.realpath('../'))
import robot_package as rb
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sympy as sym
import symengine as se
import tqdm

## Robot parameter definition

In [3]:
dh_params_sym = np.array([[0,0,0,0],
                     [0, sym.pi/2, 0, 0],
                     [0, sym.pi/2, 0, sym.pi/2],
                     [840,0,0,0],
                     [204,sym.pi/2,0,0],
                     [500,0,0,0]])

names = ['Rz_base', 'Rx_base', 'Ry_base', 'J1', 'J2', 'Dish']
types = ['fixed', 'revolute', 'revolute', 'revolute', 'revolute', 'revolute']




com_symbols = np.array([[sym.Symbol('c_x_0'), sym.Symbol('c_y_0'), sym.Symbol('c_z_0')],
                        [sym.Symbol('c_x_1'), sym.Symbol('c_y_1'), sym.Symbol('c_z_1')],
                        [sym.Symbol('c_x_2'), sym.Symbol('c_y_2'), sym.Symbol('c_z_2')],
                        [sym.Symbol('c_x_3'), sym.Symbol('c_y_3'), sym.Symbol('c_z_3')],
                        [sym.Symbol('c_x_4'), sym.Symbol('c_y_4'), sym.Symbol('c_z_4')],
                        [sym.Symbol('c_x_5'), sym.Symbol('c_y_5'), sym.Symbol('c_z_5')],
                        [sym.Symbol('c_x_6'), sym.Symbol('c_y_6'), sym.Symbol('c_z_6')],
                        [sym.Symbol('c_x_7'), sym.Symbol('c_y_7'), sym.Symbol('c_z_7')],
                        [sym.Symbol('c_x_8'), sym.Symbol('c_y_8'), sym.Symbol('c_z_8')],
                        [sym.Symbol('c_x_9'), sym.Symbol('c_y_9'), sym.Symbol('c_z_9')],
                        [sym.Symbol('c_x_10'), sym.Symbol('c_y_10'), sym.Symbol('c_z_10')],
                        [sym.Symbol('c_x_11'), sym.Symbol('c_y_11'), sym.Symbol('c_z_11')],
                        [sym.Symbol('c_x_12'), sym.Symbol('c_y_12'), sym.Symbol('c_z_12')],
                        [sym.Symbol('c_x_13'), sym.Symbol('c_y_13'), sym.Symbol('c_z_13')],
                        [sym.Symbol('c_x_14'), sym.Symbol('c_y_14'), sym.Symbol('c_z_14')],
                        [sym.Symbol('c_x_15'), sym.Symbol('c_y_15'), sym.Symbol('c_z_15')]])

com_symbols = com_symbols[:len(names)]

def symbolic_inertia_tensor(i):
    return np.array([sym.Symbol(f'I_{i}_xx'), sym.Symbol(f'I_{i}_yy'), sym.Symbol(f'I_{i}_zz'), sym.Symbol(f'I_{i}_xy'), sym.Symbol(f'I_{i}_yz'), sym.Symbol(f'I_{i}_xz')])

I_list_sym = np.array([symbolic_inertia_tensor(i) for i in range(len(names))])


link_mass_sym = np.array([sym.Symbol(f'm_{i}') for i in range(len(names))])


In [4]:
link_mass_sym.shape

(6,)

### Create a list with class instances using create_links function
It is also possible to create each link seperately

In [5]:
links = rb.create_links(names = names[:], 
                        dh_params = dh_params_sym, 
                        types = types, 
                        stl_paths = None, 
                        mass=link_mass_sym, 
                        CM = com_symbols,
                        inertia_matrix = I_list_sym,) 

c:\Users\matic\Desktop\Yaskawa\Yaskawa-project-Repo\ROBO2_venv\Lib\site-packages\pyvista\core\filters\data_object.py:179: PyVistaDeprecationWarning: The default value of `inplace` for the filter `PolyData.transform` will change in the future. Previously it defaulted to `True`, but will change to `False`. Explicitly set `inplace` to `True` or `False` to silence this warning.
  warnings.warn(msg, PyVistaDeprecationWarning)


### Use the list to create the robot class instance

In [6]:
robot = rb.Robot(links, name='Antena')

Could not compute joint frames


In [7]:
import symengine

Get all transformations

In [8]:
import dill
from tqdm import tqdm

In [9]:
robots = [robot]

In [10]:
tr_list = []
for robot in robots:
    Jp_list = []
    Jo_list = []
    R_list = []


    for i in range(1, robot.n+1):
        print(i)
        p_com = robot.links[i].CM*sym.Rational(1,1000)
        print(p_com)
        Jp_list.append(symengine.MutableDenseMatrix(robot.jacobian_v_sym2(joint = i, simplify=False, point=p_com, scale = sym.Rational(1,1000))))
        Jo_list.append(symengine.MutableDenseMatrix(robot.jacobian_w_sym(joint = i, simplify=False, scale = sym.Rational(1,1000))))
        R_list.append(symengine.MutableDenseMatrix(robot.fkine_sym(joint=i, simplify=False,scale=sym.Rational(1,1000))[0:3, 0:3].copy()))
    
    transformations = [Jp_list, Jo_list, R_list]
    tr_list.append(transformations)
    
    # with open(f'Data/Dynamics/robot_{robot.name}_transformations.pkl', 'w') as f:
    #     dill.dump(transformations, f)

1
[c_x_1/1000 c_y_1/1000 c_z_1/1000]
Computing link frames


100%|██████████| 6/6 [00:00<00:00, 141.51it/s]

2
[c_x_2/1000 c_y_2/1000 c_z_2/1000]
3
[c_x_3/1000 c_y_3/1000 c_z_3/1000]
4
[c_x_4/1000 c_y_4/1000 c_z_4/1000]
5
[c_x_5/1000 c_y_5/1000 c_z_5/1000]


In [11]:
robot.fkine_sym(joint=3, evalf=True,scale=sym.Rational(1,1000)).subs({q:0 for q in robot.q_sym})

Matrix([
[0,  0, 1,     0],
[0, -1, 0,     0],
[1,  0, 0, 21/25],
[0,  0, 0,     1]])

Bulid mass matrix

In [12]:
m_list = []
for i, robot in enumerate(robots):
    Jp_list, Jo_list, R_list = tr_list[i]
    
    n = robot.n
    M = symengine.MutableDenseMatrix(n,n, np.zeros(n**2).tolist())
    for i in tqdm(range(n)):
        m = robot.links[i+1].mass
        I = symengine.MutableDenseMatrix(3,3,robot.links[i+1].inertia_matrix.flatten())
        print('Link ', i)
        print('m: ' , m)
        print('I: ', I)
        
        I = R_list[i]@I@R_list[i].T
        M+= (Jp_list[i].T*Jp_list[i]*m + Jo_list[i].T@I@Jo_list[i])#.subs(sub_dict)
    m_list.append(M)
    # with open(f'Data/Dynamics/robot_{robot.name}_mass_matrix.pkl', 'wb') as f:
    #     dill.dump(M, f)


 20%|██        | 1/5 [00:00<00:00,  8.14it/s]

Link  0
m:  m_1
I:  [I_1_xx, I_1_xy, I_1_xz]
[I_1_xy, I_1_yy, I_1_yz]
[I_1_xz, I_1_yz, I_1_zz]



100%|██████████| 5/5 [00:00<00:00, 39.68it/s]

Link  1
m:  m_2
I:  [I_2_xx, I_2_xy, I_2_xz]
[I_2_xy, I_2_yy, I_2_yz]
[I_2_xz, I_2_yz, I_2_zz]

Link  2
m:  m_3
I:  [I_3_xx, I_3_xy, I_3_xz]
[I_3_xy, I_3_yy, I_3_yz]
[I_3_xz, I_3_yz, I_3_zz]

Link  3
m:  m_4
I:  [I_4_xx, I_4_xy, I_4_xz]
[I_4_xy, I_4_yy, I_4_yz]
[I_4_xz, I_4_yz, I_4_zz]

Link  4
m:  m_5
I:  [I_5_xx, I_5_xy, I_5_xz]
[I_5_xy, I_5_yy, I_5_yz]
[I_5_xz, I_5_yz, I_5_zz]



In [13]:
M

[I_1_yy + m_1*(((1/1000)*sin(q1)*c_z_1 + (1/1000)*cos(q1)*c_x_1)**2 + ((-1/1000)*sin(q1)*c_x_1 + (1/1000)*cos(q1)*c_z_1)**2) + m_2*(((1/1000)*cos(q1)*c_y_2 + (-1/1000)*sin(q1)*cos(q2)*c_z_2 + (1/1000)*sin(q2)*sin(q1)*c_x_2)**2 + ((1/1000)*sin(q1)*c_y_2 + (-1/1000)*sin(q2)*cos(q1)*c_x_2 + (1/1000)*cos(q1)*cos(q2)*c_z_2)**2) + m_3*(((1/1000)*c_x_3*(sin(q1)*sin(q3) - sin(q2)*cos(q1)*cos(q3)) + (1/1000)*c_y_3*(sin(q1)*cos(q3) + sin(q2)*sin(q3)*cos(q1)) + (21/25)*sin(q1)*sin(q3) + (-21/25)*sin(q2)*cos(q1)*cos(q3) + (1/1000)*cos(q1)*cos(q2)*c_z_3)**2 + ((-1/1000)*c_x_3*(-sin(q3)*cos(q1) - sin(q2)*sin(q1)*cos(q3)) + (-1/1000)*c_y_3*(-cos(q1)*cos(q3) + sin(q2)*sin(q1)*sin(q3)) + (21/25)*sin(q3)*cos(q1) + (-1/1000)*sin(q1)*cos(q2)*c_z_3 + (21/25)*sin(q2)*sin(q1)*cos(q3))**2) + m_4*(((1/1000)*(sin(q4)*(sin(q1)*sin(q3) - sin(q2)*cos(q1)*cos(q3)) - cos(q4)*(sin(q1)*cos(q3) + sin(q2)*sin(q3)*cos(q1)))*c_z_4 + (1/1000)*(sin(q4)*(sin(q1)*cos(q3) + sin(q2)*sin(q3)*cos(q1)) + cos(q4)*(sin(q1)*sin(q3) -

In [14]:
q = robot.q_sym
dq = robot.dq_sym

In [15]:
q[0]

q1

In [16]:
# calculate derivatives of M with respect to q
dm_list = []
for i, robot in enumerate(robots):
    M = m_list[i]
    dM_i = []
    for i in tqdm(range(robot.n)):
        dM_i.append(symengine.diff(M, q[i]))
    # with open(f'Data/Dynamics/robot_{robot.name}_dM_dq.pkl', 'wb') as f:
    #     dill.dump(dM_i, f)
    dm_list.append(dM_i)



100%|██████████| 5/5 [00:00<00:00, 126.80it/s]


Build coriolliss matrix

In [17]:
C_list = []
for i, robot in enumerate(robots):
    dM_i = dm_list[i]

    n = robot.n
    C = symengine.MutableDenseMatrix(n,n, np.zeros(n**2).tolist())
    C_IJK = np.zeros((n,n,n), dtype=object)
    for i in tqdm(range(n)):
        for j in range(n):
            for k in range(n):
                C_IJK[i,j,k] += 0.5*(dM_i[k][i,j] + dM_i[j][i,k] - dM_i[i][j,k])
                C[i,j] += C_IJK[i,j,k]*dq[k]
    C_list.append(C)

100%|██████████| 5/5 [00:00<?, ?it/s]


Get gravity torques

In [ ]:
g_list = []
for i, robot in enumerate(robots):
    n = robot.n
    g = symengine.MutableDenseMatrix(n,1, np.zeros(n).tolist())
    g_sym = symengine.Symbol('g')
    g_sym = -9.81
    g_vec = symengine.MutableDenseMatrix(3,1, [0,0,g_sym])
    for i in tqdm(range(robot.n)):
        m = robot.links[i+1].mass
        g += (Jp_list[i].T*m*g_vec)
    g_list.append(g)

100%|██████████| 5/5 [00:00<?, ?it/s]


In [18]:
params = np.array([*link_mass_sym, *com_symbols.flatten(), *I_list_sym.flatten()])

In [24]:
cmg_list = []

dir = 'Data/Antena/Dynamics/'
for i, robot in tqdm(enumerate(robots)):
    C = C_list[i]
    M = m_list[i]
    M_inv = M.inv()
    g = g_list[i]
    M_fun = symengine.lambdify([*q, *params], M, real = True, cse = True)
    print('M done')
    M_inv_fun = symengine.lambdify([*q, *params], M_inv, real = True, cse = True)
    print('M_inv done')
    g_fun = symengine.lambdify([*q, *params], g, real = True, cse = True)
    print('g done')
    C_fun = symengine.lambdify([*q, *dq, *params], C, real = True, cse = True)
    print('C done')
    cmg_list.append([C_fun, M_fun, M_inv_fun, g_fun])
    

0it [00:00, ?it/s]

M done
M_inv done
g done


1it [00:00,  1.87it/s]

C done


In [25]:
m_list.shape

AttributeError: 'list' object has no attribute 'shape'

In [26]:
name = 'Antena'
dir = 'Data/Antena/Dynamics/'
with open(f'{dir}M_fun_{name}.pkl', 'wb') as f:
    dill.dump(M_fun, f)
with open(f'{dir}M_inv_fun_{name}.pkl', 'wb') as f:
    dill.dump(M_inv_fun, f)
with open(f'{dir}g_fun_{name}.pkl', 'wb') as f:
    dill.dump(g_fun, f)
with open(f'{dir}C_fun_{name}.pkl', 'wb') as f:
    dill.dump(C_fun, f)

In [27]:
name = robot.name
dir = 'Data/Antena/Dynamics/'
with open(f'{dir}M_fun_{name}.pkl', 'rb') as f:
    M_fun = dill.load(f)
with open(f'{dir}M_inv_fun_{name}.pkl', 'rb') as f:
    M_inv_fun = dill.load(f)
with open(f'{dir}g_fun_{name}.pkl', 'rb') as f:
    g_fun = dill.load(f)
with open(f'{dir}C_fun_{name}.pkl', 'rb') as f:
    C_fun = dill.load(f)